# WideResNet50-2 PatchCore Dataset Helper

This helper notebook prepares the labeled local split used by the current best anomaly detector.

Requested split:
- train: `120,000` total with `6,000` anomalies
- validation: `10,000` total with `500` anomalies
- test: `20,000` total with exactly `1,000` anomalies

In [ ]:
import os
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
BUNDLE_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "helpers" / "patchcore_wrn50_local.py").exists():
        BUNDLE_ROOT = candidate
        break

if BUNDLE_ROOT is None:
    raise RuntimeError("Could not locate the local WRN PatchCore experiment root.")

HELPERS_ROOT = BUNDLE_ROOT / "helpers"
if str(HELPERS_ROOT) not in sys.path:
    sys.path.insert(0, str(HELPERS_ROOT))

from patchcore_wrn50_local import (
    DEFAULT_SPLIT_CONFIG,
    auto_find_raw_pickle,
    defect_type_summary,
    load_wafer_array,
    metadata_paths,
    prepare_dataset,
    resolve_data_root,
    resolve_output_root,
    set_seed,
    split_summary,
    split_summary_wide,
)

In [ ]:
RAW_PICKLE = os.environ.get("WM811K_RAW_PICKLE")
IMAGE_SIZE = 64
SEED = 42
SPLIT_CONFIG = DEFAULT_SPLIT_CONFIG.copy()
DATA_ROOT = resolve_data_root(BUNDLE_ROOT)
OUTPUT_ROOT = resolve_output_root(BUNDLE_ROOT)

set_seed(SEED)
raw_pickle = auto_find_raw_pickle(RAW_PICKLE)
metadata_path = prepare_dataset(raw_pickle, DATA_ROOT, IMAGE_SIZE, SPLIT_CONFIG, seed=SEED, overwrite=False)
metadata = pd.read_csv(metadata_path)
_, arrays_dir = metadata_paths(DATA_ROOT, IMAGE_SIZE, SPLIT_CONFIG)

display(split_summary(metadata))
display(split_summary_wide(metadata))
display(defect_type_summary(metadata).head(20))

print("Bundle root:", BUNDLE_ROOT)
print("Data root:", DATA_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Raw pickle:", raw_pickle)
print("Metadata path:", metadata_path)
print("Arrays dir:", arrays_dir)

In [ ]:
sample_specs = [
    ("train", 0, "Train normal"),
    ("train", 1, "Train anomaly"),
    ("val", 0, "Val normal"),
    ("val", 1, "Val anomaly"),
    ("test", 0, "Test normal"),
    ("test", 1, "Test anomaly"),
]

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, (split_name, is_anomaly, title) in zip(axes.ravel(), sample_specs):
    rows = metadata[(metadata["split"] == split_name) & (metadata["is_anomaly"] == is_anomaly)]
    row = rows.iloc[0]
    wafer = load_wafer_array(DATA_ROOT, row["array_path"])
    ax.imshow(wafer, cmap="viridis")
    ax.set_title(f"{title}\n{row['defect_type']}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
summary_df = split_summary_wide(metadata)
observed = summary_df.set_index("split")
expected = {
    "train": {"count": SPLIT_CONFIG["train_total"], "anomalies": SPLIT_CONFIG["train_anomalies"]},
    "val": {"count": SPLIT_CONFIG["val_total"], "anomalies": SPLIT_CONFIG["val_anomalies"]},
    "test": {"count": SPLIT_CONFIG["test_total"], "anomalies": SPLIT_CONFIG["test_anomalies"]},
}

for split_name, values in expected.items():
    assert int(observed.loc[split_name, "count"]) == int(values["count"])
    assert int(observed.loc[split_name, "anomalies"]) == int(values["anomalies"])

helper_output_dir = OUTPUT_ROOT / "dataset_helper"
helper_output_dir.mkdir(parents=True, exist_ok=True)
summary_path = helper_output_dir / "split_summary.json"
summary_payload = {
    "raw_pickle": str(raw_pickle),
    "metadata_path": str(metadata_path),
    "arrays_dir": str(arrays_dir),
    "split_config": SPLIT_CONFIG,
    "split_summary": summary_df.to_dict(orient="records"),
}
summary_path.write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")
print("Saved helper summary to", summary_path)